# Logistic Regression — Math Foundations

Logistic regression adapts linear regression for **binary classification** by squashing
the output through the sigmoid function so predictions are probabilities in $[0, 1]$.

**What we derive:**
- The sigmoid (logistic) function and why it's used
- The decision boundary
- Cross-entropy loss (log-loss) — why not MSE for classification?
- Gradient of cross-entropy w.r.t. $\boldsymbol{\theta}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../..')
from src.utils import set_style, sigmoid

set_style()
np.random.seed(42)

## 1. The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Properties:
- Output always in $(0, 1)$ — interpretable as a probability
- $\sigma(0) = 0.5$ — decision boundary at $z = 0$
- $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ — elegant derivative

In [ ]:
z = np.linspace(-8, 8, 300)
sig_z  = sigmoid(z)
dsig_z = sig_z * (1 - sig_z)   # derivative

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sigmoid
axes[0].plot(z, sig_z, color='#2563EB', lw=2.5)
axes[0].axhline(0.5, color='#DC2626', linestyle='--', lw=1.2, label='σ(0) = 0.5')
axes[0].axvline(0,   color='#DC2626', linestyle=':',  lw=1.2)
axes[0].fill_between(z[z<0], sig_z[z<0], 0.5, alpha=0.08, color='#DC2626', label='Class 0 region')
axes[0].fill_between(z[z>0], sig_z[z>0], 0.5, alpha=0.08, color='#2563EB', label='Class 1 region')
axes[0].set_xlabel('z = θᵀx')
axes[0].set_ylabel('σ(z)')
axes[0].set_title('Sigmoid Function')
axes[0].legend()

# Derivative
axes[1].plot(z, dsig_z, color='#16A34A', lw=2.5)
axes[1].set_xlabel('z')
axes[1].set_ylabel("σ'(z) = σ(z)(1-σ(z))")
axes[1].set_title('Sigmoid Derivative')

plt.tight_layout()
plt.show()

## 2. Hypothesis

$$h_{\boldsymbol{\theta}}(\mathbf{x}) = \sigma(\boldsymbol{\theta}^T \mathbf{x}) = P(y=1 \mid \mathbf{x}; \boldsymbol{\theta})$$

**Decision boundary:** Predict class 1 if $h > 0.5$, i.e., if $\boldsymbol{\theta}^T \mathbf{x} > 0$.

In [ ]:
# 2-D toy dataset: two Gaussian clouds
n, d = 200, 2
X0 = np.random.multivariate_normal([1, 1],  np.eye(2) * 0.5, n // 2)
X1 = np.random.multivariate_normal([-1, -1], np.eye(2) * 0.5, n // 2)
X_data = np.vstack([X0, X1])
y_data = np.hstack([np.ones(n // 2), np.zeros(n // 2)])

# True decision boundary: θ = [-1, 1, 1] (intercept, w1, w2)
theta_true = np.array([0.0, 1.0, 1.0])

# Plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X0[:, 0], X0[:, 1], color='#2563EB', alpha=0.5, label='Class 1', s=30)
ax.scatter(X1[:, 0], X1[:, 1], color='#DC2626', alpha=0.5, label='Class 0', s=30)

# Decision boundary: θ₀ + θ₁x₁ + θ₂x₂ = 0  →  x₂ = -(θ₀ + θ₁x₁)/θ₂
x1_vals = np.linspace(-3, 3, 100)
x2_vals = -(theta_true[0] + theta_true[1] * x1_vals) / theta_true[2]
ax.plot(x1_vals, x2_vals, 'k--', lw=2, label='Decision boundary')

ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Logistic Regression — Decision Boundary')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Why Not MSE? — The Cross-Entropy Loss

Using MSE with sigmoid produces a **non-convex** cost surface (many local minima).

Maximum Likelihood Estimation gives us the convex **cross-entropy (log-loss)**:

$$J(\boldsymbol{\theta}) = -\frac{1}{m}\sum_{i=1}^{m} \left[ y^{(i)} \log h^{(i)} + (1 - y^{(i)}) \log(1 - h^{(i)}) \right]$$

Intuition:
- If $y=1$ and $h \to 1$: $\log(1) = 0$ → zero loss
- If $y=1$ and $h \to 0$: $\log(0) \to -\infty$ → huge penalty

In [ ]:
h   = np.linspace(0.001, 0.999, 300)
loss_y1 = -np.log(h)       # cost when y=1
loss_y0 = -np.log(1 - h)   # cost when y=0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(h, loss_y1, color='#2563EB', lw=2)
axes[0].set_xlabel('Predicted probability h(x)')
axes[0].set_ylabel('Cost')
axes[0].set_title('Cost when y = 1  →  −log(h)')

axes[1].plot(h, loss_y0, color='#DC2626', lw=2)
axes[1].set_xlabel('Predicted probability h(x)')
axes[1].set_ylabel('Cost')
axes[1].set_title('Cost when y = 0  →  −log(1−h)')

plt.tight_layout()
plt.show()

## 4. Gradient of Cross-Entropy (Derivation Sketch)

After applying the chain rule through the sigmoid:

$$\frac{\partial J}{\partial \boldsymbol{\theta}} = \frac{1}{m} X^T (\hat{\mathbf{y}} - \mathbf{y})$$

where $\hat{y}^{(i)} = \sigma(\boldsymbol{\theta}^T \mathbf{x}^{(i)})$.

Remarkably, this has the **same form** as the linear regression gradient — only
$\hat{y}$ is now produced by the sigmoid instead of a linear function.

In [ ]:
def cross_entropy_cost(X, y, theta):
    """J(θ) = -(1/m) [y·log(h) + (1-y)·log(1-h)]  (numerically stable)"""
    m   = len(y)
    h   = sigmoid(X @ theta)
    eps = 1e-15   # clip to avoid log(0)
    h   = np.clip(h, eps, 1 - eps)
    return -float(y @ np.log(h) + (1 - y) @ np.log(1 - h)) / m


def logistic_gradient(X, y, theta):
    """∂J/∂θ = (1/m) Xᵀ(h - y)"""
    m = len(y)
    h = sigmoid(X @ theta)
    return X.T @ (h - y) / m


# Quick sanity check
X_test_mat = np.column_stack([np.ones(n), X_data])
theta_init = np.zeros(3)
print(f'Initial cost   : {cross_entropy_cost(X_test_mat, y_data, theta_init):.4f}')
print(f'Initial gradient (first 3): {logistic_gradient(X_test_mat, y_data, theta_init)}')

## Summary

| Concept | Formula |
|---------|--------|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ |
| Hypothesis | $\hat{y} = \sigma(X\theta)$ |
| Cross-entropy loss | $J = -\frac{1}{m}[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| Gradient | $\frac{\partial J}{\partial \theta} = \frac{1}{m}X^T(\hat{y}-y)$ |
| Decision boundary | $\theta^Tx = 0$ |

**Next:** [Implement logistic regression from scratch →](02_from_scratch_numpy.ipynb)